In [1]:
from ingest import load_faq_data,build_index
from rag_helper import RagHelper

from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
openai_client = OpenAI()

In [2]:
docs = load_faq_data()
index = build_index(docs)

In [4]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [5]:
## define the tool
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
## prepare tool description
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [10]:
result = search("when course starts")

In [16]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration"}
function_call: search {"query":"course FAQ enrollment late join discovered course join it"}


In [17]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration"}', call_id='call_fPyyk3qXcGQKz62eb4IXF4R5', name='search', type='function_call', id='fc_0c5133270f96aa4a006a2f52fbf07c8190b5a19964b32d9911', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course FAQ enrollment late join discovered course join it"}', call_id='call_NZr

In [19]:
it = 1
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

while True:
    print(f"iteration #{it}...context len:{len(messages)}")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...context len:2
function_call: search {"query":"join course discovered late can I join enrollment late FAQ"}
iteration #2...context len:4
function_call: search {"query":"self-paced mode certificate join course late project submissions peer review live cohort FAQ"}
iteration #3...context len:6
ASSISTANT:
Yes — you can still join.

If your goal is to get a certificate, make sure you submit your project while submissions are still open. Certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help clarify how the certificate/project/peer-review process works.


In [23]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer, messages

In [24]:
ans, msg = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local usage start server model pull run"}
iteration #2...
function_call: search {"query":"Ollama local server localhost 11434 ollama run llama3 python client install ollama model pull"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download and install the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat prompt.

3. **Check the local server**
   In another terminal:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response indicating the Ollama server is running.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       mod

In [25]:
msg

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"Ollama run locally install local usage start server model pull run"}', call_id='call_wqytXhlrZzMX6CjWSPI2di99', name='search', type='function_call', id='fc_042c3dfd5eeed27a006a2f548a2bd88190a0f884393c6da7aa', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_wqytXhlrZzMX6CjWSPI2di99',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run local install setup model server"}
function_call: search {"query":"Ollama local run install start server models"}
function_call: search {"query":"local Ollama FAQ running locally"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` installer from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',

("To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` installer from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model='llama3',\n       messages=[{'role': 'user', 'content': 'Hello!'}]\n   )\n\n   print(response['message']['content'])\n   ```\n\nIf you’re in a notebook or need to restart the server, you can also run:\n```bash\nollama serve\n```\nOr in some notebook

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course FAQ late join enrollment discovered course"}
function_call: search {"query":"can I still join the course after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open. If you’d like, I can also help clarify what’s needed for the certificate or how to catch up quickly. Are there other areas you want to explore?


('Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still open. If you’d like, I can also help clarify what’s needed for the certificate or how to catch up quickly. Are there other areas you want to explore?',
 [{'role': 'developer',
   'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
  {'role': 'user',
   'content': 'I just discovered the course. Can I still join it?'},
  ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered course can I still join"}', c

In [29]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
function_call: search {"query":"what is the queen gambit queen's gambit opening"}
iteration #2...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn as a “gambit” to try to gain control of the center. Despite the name, it’s usually **not a true sacrifice**, because Black often can’t safely keep the pawn.

A simple idea behind it:
- White uses the c-pawn to challenge Black’s d-pawn
- aims for strong central control
- often leads to solid, strategic positions rather than quick attacks

There are two main forms:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain the difference between **Queen’s Gambit Accepted** and **Declined** in an easy way.


('The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn as a “gambit” to try to gain control of the center. Despite the name, it’s usually **not a true sacrifice**, because Black often can’t safely keep the pawn.\n\nA simple idea behind it:\n- White uses the c-pawn to challenge Black’s d-pawn\n- aims for strong central control\n- often leads to solid, strategic positions rather than quick attacks\n\nThere are two main forms:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain the difference between **Queen’s Gambit Accepted** and **Declined** in an easy way.',
 [{'role': 'developer',
   'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when 

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit chess"}
iteration #4...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question. If you meant something course-related, try rephrasing it with the course topic or term.

Is there another area you want to explore?


('I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question. If you meant something course-related, try rephrasing it with the course topic or term.\n\nIs there another area you want to explore?',
 [{'role': 'developer',
   'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user

In [3]:
!uv add toyaikit

Resolved 127 packages in 1.41s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/78.27 KiB           
⠙ Preparing packages... (0/7)-------------------     0 B/78.27 KiB           
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/78.27 KiB           
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.27 KiB         
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.27 KiB         
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.27 KiB         
docstring-parser     

In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

NameError: name 'search_tool' is not defined

In [33]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [35]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [37]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [38]:
result.cost

CostInfo(input_cost=Decimal('0.00149475'), output_cost=Decimal('0.001467'), total_cost=Decimal('0.00296175'))

In [40]:
context_buffer = result.all_messages

In [42]:
result2 = runner.loop(
    prompt="Do I need docker for it?",
    previous_messages=context_buffer,
    callback=callback,
)

-> Response received


-> Response received


In [43]:
result2

LoopResult(new_messages=[EasyInputMessage(content='Do I need docker for it?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Ollama docker required local install faq"}', call_id='call_u9cPAQPiQpb59xa5i0z76LGH', name='search', type='function_call', id='fc_08e0fc4c8ce55dbf006a30a4f35f20819182e9bb5955b4800b', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"Ollama Docker FAQ local run without docker"}', call_id='call_hOjgh1OssVihIDux5976CrFp', name='search', type='function_call', id='fc_08e0fc4c8ce55dbf006a30a4f35f3081919e51c7a8fc966170', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_u9cPAQPiQpb59xa5i0z76LGH', 'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: Introduction to LLMs and RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://oll

In [44]:
result2.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Ollama local run install start server FAQ"}', call_id='call_QqtxI6rRqBuVVMb42R0CInjm', name='search', type='function_call', id='fc_08e0fc4c8ce55dbf006a30a450b25c81918612c0381c8341b1', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally install command faq"}', call_id='call

In [45]:
runner.run()